# samIRA: Vigilancia Epidemiológica y Pronóstico de Infecciones Respiratorias Agudas
## Cuaderno 02: Modelado Predictivo, Baselines, Optimización y Evaluación Causal
**Autor:** Aldair Maquera & Equipo samIRA  
**Región de Estudio:** Departamento de Puno, Perú (13 Provincias)  
**Horizontes:** $h = 1$ semana y $h = 4$ semanas adelante  
**Partición Temporal:** Entrenamiento ($\le 2022$), Validación (2023–2024), Prueba (2025)

---

### 1. Objetivos y Marco Metodológico
1. **Modelos Baselines de Referencia:** Implementar y evaluar modelos de referencia estándar de series temporales:
   - *Naive:* Proyecta el último valor observado en el origen $t$.
   - *Seasonal Naive:* Proyecta el valor de la misma semana epidemiológica del año anterior ($y-1, w_{t+h}$), respetando años con semana 53.
   - *ETS / Holt-Winters:* Suavizamiento exponencial amortiguado con ciclo estacional anual de 52 períodos.
2. **Modelos de Aprendizaje Automático y Optimización:**
   - *LightGBM Global* y *XGBoost Global:* Ajustados sobre el panel combinado de las 13 provincias.
   - *Búsqueda y Optimización de Hiperparámetros:* Búsqueda en cuadrícula evaluando múltiples configuraciones sobre la partición de validación temporal (2023–2024) minimizando el WAPE (%). Comparación formal entre configuración inicial por defecto vs configuración optimizada.
   - *LightGBM Independiente:* Modelos locales ajustados por provincia individual.
3. **Corrección Metodológica Crítica ($h = 4$):**
   - En auditorías previas se identificó una contaminación temporal en $h = 4$: 52 observaciones (las semanas 1 a 4 de 2023 en las 13 provincias) se filtraban erróneamente en el entrenamiento al basar el corte en el año de origen $t$ (semanas 49–52 de 2022) en lugar del año de la variable objetivo $y_{t+4}$.
   - **Corrección obligatoria:** Las particiones de entrenamiento, validación y prueba se segmentan estrictamente en función del año de la variable objetivo (`target_ano`).
   - Las armónicas de Fourier se calculan en base a la semana objetivo $w_{t+h}$.
   - Se incorporan verificaciones con `assert` que imposibilitan cualquier fuga de información.
4. **Intervalos de Predicción:**
   - Intervalos al 95% calibrados empíricamente en la partición de validación (2023–2024) y evaluados por provincia y globalmente.
5. **Transparencia sobre el Período 2025:** Se documenta formalmente que los datos de 2025 han sido inspeccionados en fases previas y deben interpretarse como una prueba semi-ciega out-of-sample tras las correcciones metodológicas.

### 2. Configuración del Entorno, Librerías y Semilla Aleatoria

In [1]:
import os
import sys

# Asegurar que el directorio de trabajo sea la raíz del repositorio
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Directorio de trabajo actual:", os.getcwd())

import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 140

RANDOM_SEED = 42
ANIOS_SEMANA_53 = {2003, 2008, 2014, 2020, 2025}

def get_official_weeks_in_year(ano: int) -> int:
    return 53 if ano in ANIOS_SEMANA_53 else 52

def compute_metrics(y_true, y_pred):
    # Calcula MAE, RMSE, WAPE (%) y Bias
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    yt = y_true[mask]
    yp = y_pred[mask]
    if len(yt) == 0:
        return {"mae": np.nan, "rmse": np.nan, "wape_pct": np.nan, "bias": np.nan, "n": 0}
    mae = mean_absolute_error(yt, yp)
    rmse = root_mean_squared_error(yt, yp)
    sum_yt = np.sum(yt)
    wape = (np.sum(np.abs(yt - yp)) / sum_yt * 100) if sum_yt > 0 else np.nan
    bias = np.mean(yp - yt)
    return {
        "mae": round(float(mae), 3),
        "rmse": round(float(rmse), 3),
        "wape_pct": round(float(wape), 3),
        "bias": round(float(bias), 3),
        "n": int(len(yt))
    }

print("Librerías y funciones de evaluación inicializadas exitosamente.")

Directorio de trabajo actual: F:\dxp-projects\DevBasics\R y statistics\samIRA


Librerías y funciones de evaluación inicializadas exitosamente.


### 3. Carga y Verificación de los Paneles Curados
Se leen los paneles semanal provincial y regional generados en el Cuaderno 01.

In [2]:
prov_path = os.path.join("data", "processed", "panel_semanal_provincial_puno.parquet")
reg_path = os.path.join("data", "processed", "panel_semanal_regional_puno.parquet")

assert os.path.exists(prov_path), f"No existe el archivo provincial: {prov_path}"
assert os.path.exists(reg_path), f"No existe el archivo regional: {reg_path}"

df_prov = pd.read_parquet(prov_path)
df_reg = pd.read_parquet(reg_path)

print(f"Panel Provincial: {df_prov.shape[0]:,} filas, {df_prov.shape[1]} columnas.")
print(f"Panel Regional:   {df_reg.shape[0]:,} semanas.")

# Verificación de ausencias originales (153 ausencias con NaN estricto)
n_faltantes = df_prov["ira_no_neumonia"].isna().sum()
n_no_eval = (~df_prov["es_evaluable"]).sum()
assert n_faltantes == 153, f"Se esperaban 153 ausencias con NaN, se encontraron {n_faltantes}"
assert n_no_eval == 153, f"Se esperaban 153 semanas no evaluables, se encontraron {n_no_eval}"
print(f">>> Consistencia validada: {n_faltantes} semanas de ausencia con NaN estricto y es_evaluable=False.")

Panel Provincial: 17,641 filas, 13 columnas.
Panel Regional:   1,357 semanas.
>>> Consistencia validada: 153 semanas de ausencia con NaN estricto y es_evaluable=False.


### 4. Ajuste y Evaluación de Modelos Baseline (Naive, Seasonal Naive, ETS)
Se evalúan para horizontes $h=1$ y $h=4$ semanas:
- **Naive:** $y_{t+h|t} = y_t$ (último valor conocido; si $y_t$ es NaN, se utiliza LOCF retrospectivo causal).
- **Seasonal Naive:** $y_{t+h|t} = y_{target\_ano - 1, target\_semana}$ (valor homólogo del año anterior, mapeando la semana 53 a la 52 si el año precedente carecía de semana 53).
- **ETS (Holt-Winters):** Suavizamiento exponencial con estacionalidad anual de 52 períodos ajustado sobre los datos de entrenamiento ($\le 2022$) y proyectado rolling desde el origen $t$.

In [3]:
prov_lookup = df_prov.set_index(["provincia", "ano", "semana"])["ira_no_neumonia"].to_dict()
reg_lookup = df_reg.set_index(["ano", "semana"])["ira_no_neumonia"].to_dict()

baseline_preds = []
provincias = sorted(df_prov["provincia"].unique())
horizons = [1, 4]

for prov in provincias:
    p_df = df_prov[df_prov["provincia"] == prov].sort_values(by=["ano", "semana"]).reset_index(drop=True)
    causal_series = p_df["ira_no_neumonia"].ffill()
    
    # Ajustar ETS sobre entrenamiento (<= 2022)
    train_mask = p_df["ano"] <= 2022
    train_series = p_df.loc[train_mask, "ira_no_neumonia"].ffill().bfill()
    try:
        ets_model = ExponentialSmoothing(
            train_series.values,
            trend="add",
            damped_trend=True,
            seasonal="add",
            seasonal_periods=52,
            initialization_method="estimated"
        ).fit()
        alpha = ets_model.params.get("smoothing_level")
        beta = ets_model.params.get("smoothing_trend")
        gamma = ets_model.params.get("smoothing_seasonal")
        phi = ets_model.params.get("damping_trend", 0.95)
        ets_fitted = True
    except Exception as e:
        ets_fitted = False

    for h in horizons:
        for idx in range(len(p_df)):
            origin_idx = idx - h
            if origin_idx < 0:
                continue
                
            target_row = p_df.iloc[idx]
            target_year = int(target_row["ano"])
            target_week = int(target_row["semana"])
            target_val = target_row["ira_no_neumonia"]
            is_eval = bool(target_row["es_evaluable"])
            
            if target_year in [2023, 2024]:
                part = "val"
            elif target_year == 2025:
                part = "test"
            else:
                continue
                
            # 1. Naive
            pred_naive = max(0.0, float(causal_series.iloc[origin_idx]))
            
            # 2. Seasonal Naive
            prev_year = target_year - 1
            lookup_week = 52 if (target_week == 53 and get_official_weeks_in_year(prev_year) == 52) else target_week
            pred_snaive = prov_lookup.get((prov, prev_year, lookup_week), np.nan)
            if np.isnan(pred_snaive):
                pred_snaive = pred_naive
            pred_snaive = max(0.0, float(pred_snaive))
            
            # 3. ETS Forecast
            pred_ets = np.nan
            if ets_fitted:
                try:
                    s_up_to_origin = p_df.loc[:origin_idx, "ira_no_neumonia"].ffill().bfill().values
                    m_step = ExponentialSmoothing(
                        s_up_to_origin,
                        trend="add",
                        damped_trend=True,
                        seasonal="add",
                        seasonal_periods=52,
                        initialization_method="estimated"
                    ).fit(smoothing_level=alpha, smoothing_trend=beta, smoothing_seasonal=gamma, damping_trend=phi, use_brute=False)
                    fc = m_step.forecast(h)
                    pred_ets = max(0.0, float(np.asarray(fc)[-1]))
                except:
                    pred_ets = pred_snaive

            for mod_name, p_val in [("Naive", pred_naive), ("Seasonal Naive", pred_snaive), ("ETS / Holt-Winters", pred_ets)]:
                baseline_preds.append({
                    "ambito": "PROVINCIAL",
                    "entidad": prov,
                    "target_ano": target_year,
                    "target_semana": target_week,
                    "ano_semana": target_row["ano_semana"],
                    "horizonte_h": h,
                    "particion": part,
                    "es_evaluable": is_eval,
                    "y_true": target_val,
                    "modelo": mod_name,
                    "y_pred": p_val
                })

df_base_preds = pd.DataFrame(baseline_preds)
print(f"Predicciones de Baselines generadas: {len(df_base_preds):,} filas.")
df_base_preds.head()

Predicciones de Baselines generadas: 12,246 filas.


,ambito,entidad,target_ano,target_semana,ano_semana,horizonte_h,particion,es_evaluable,y_true,modelo,y_pred
0,PROVINCIAL,AZANGARO,2023,1,2023-W01,1,val,True,102.0,Naive,100.000000
1,PROVINCIAL,AZANGARO,2023,1,2023-W01,1,val,True,102.0,Seasonal Naive,118.000000
2,PROVINCIAL,AZANGARO,2023,1,2023-W01,1,val,True,102.0,ETS / Holt-Winters,97.482396
3,PROVINCIAL,AZANGARO,2023,2,2023-W02,1,val,True,99.0,Naive,102.000000
4,PROVINCIAL,AZANGARO,2023,2,2023-W02,1,val,True,99.0,Seasonal Naive,123.000000


### 5. Ingeniería de Características Causal y Corrección Metodológica para $h=1$ y $h=4$
#### Reglas Causales Estrictas:
1. **Indexación por Objetivo:** Cada observación predictiva se etiqueta inequívocamente con `target_ano` y `target_semana`.
2. **Causalidad Temporal:** Para un horizonte $h$, en la fila que predice $y_{t+h}$, las variables predictoras utilizan estrictamente información observada en el origen $t$ o antes ($y_t, y_{t-1}, \dots$).
3. **Armónicas de Fourier Corregidas:** $\sin(2\pi w_{target}/52)$ y $\cos(2\pi w_{target}/52)$ se calculan sobre la **semana objetivo** $w_{target}$, eliminando el desfasaje estacional previo.
4. **Retardo Estacional Homólogo Causal:** Corresponde al valor en la semana homóloga del año previo ($target\_ano - 1, target\_semana$).
5. **División Temporal Estricta:**
   - Entrenamiento: $target\_ano \le 2022$ (2001–2022).
   - Validación: $target\_ano \in [2023, 2024]$.
   - Prueba: $target\_ano == 2025$.
   - **Cero solapamiento temporal.**

In [4]:
def build_causal_features(df_input: pd.DataFrame, horizon: int) -> pd.DataFrame:
    # Construye el conjunto de características predictivas para un horizonte h garantizando causalidad estricta.
    prov_lookup = df_input.set_index(["provincia", "ano", "semana"])["ira_no_neumonia"].to_dict()
    records = []
    
    for prov, grp in df_input.groupby("provincia", sort=False):
        grp = grp.copy().sort_values(by=["ano", "semana"]).reset_index(drop=True)
        raw_series = grp["ira_no_neumonia"].copy()
        # Soporte causal para retardos (LOCF retrospectivo)
        support = raw_series.ffill().bfill()
        
        n_rows = len(grp)
        for idx in range(n_rows):
            origin_idx = idx - horizon
            if origin_idx < 0:
                continue
                
            target_row = grp.iloc[idx]
            target_year = int(target_row["ano"])
            target_week = int(target_row["semana"])
            target_val = target_row["ira_no_neumonia"]
            is_eval = bool(target_row["es_evaluable"])
            
            origin_row = grp.iloc[origin_idx]
            origin_year = int(origin_row["ano"])
            origin_week = int(origin_row["semana"])
            
            # Retardo estacional homólogo del año anterior
            prev_year = target_year - 1
            lookup_w = 52 if (target_week == 53 and get_official_weeks_in_year(prev_year) == 52) else target_week
            s_val = prov_lookup.get((prov, prev_year, lookup_w), np.nan)
            if np.isnan(s_val):
                s_val = support.iloc[origin_idx]
                
            # Lags autorregresivos causales desde origin_idx hacia atrás
            lag1 = support.iloc[origin_idx]
            lag2 = support.iloc[origin_idx - 1] if origin_idx >= 1 else lag1
            lag3 = support.iloc[origin_idx - 2] if origin_idx >= 2 else lag2
            lag4 = support.iloc[origin_idx - 3] if origin_idx >= 3 else lag3
            lag8 = support.iloc[origin_idx - 7] if origin_idx >= 7 else lag4
            
            # Rolling causal retrospectivo (hasta origin_idx inclusive)
            r_window_4 = support.iloc[max(0, origin_idx - 3) : origin_idx + 1]
            r_window_13 = support.iloc[max(0, origin_idx - 12) : origin_idx + 1]
            
            mean_4 = r_window_4.mean()
            std_4 = r_window_4.std() if len(r_window_4) > 1 else 0.0
            if np.isnan(std_4):
                std_4 = 0.0
                
            mean_13 = r_window_13.mean()
            std_13 = r_window_13.std() if len(r_window_13) > 1 else 0.0
            if np.isnan(std_13):
                std_13 = 0.0
                
            # Armónicas de Fourier calculadas SOBRE LA SEMANA OBJETIVO
            sin_fourier = np.sin(2 * np.pi * target_week / 52.0)
            cos_fourier = np.cos(2 * np.pi * target_week / 52.0)
            
            # Indicador de pandemia
            es_pandemia = 1 if (2020 <= target_year <= 2022) else 0
            
            records.append({
                "provincia": prov,
                "origin_ano": origin_year,
                "origin_semana": origin_week,
                "target_ano": target_year,
                "target_semana": target_week,
                "ano_semana": target_row["ano_semana"],
                "horizonte_h": horizon,
                "es_evaluable": is_eval,
                "y_true": target_val,
                "lag_1": lag1,
                "lag_2": lag2,
                "lag_3": lag3,
                "lag_4": lag4,
                "lag_8": lag8,
                "rolling_mean_4": mean_4,
                "rolling_std_4": std_4,
                "rolling_mean_13": mean_13,
                "rolling_std_13": std_13,
                "seasonal_lag_1y": s_val,
                "sin_semana": sin_fourier,
                "cos_semana": cos_fourier,
                "es_pandemia": es_pandemia
            })
            
    df_feat = pd.DataFrame(records)
    df_feat["provincia"] = df_feat["provincia"].astype("category")
    return df_feat

df_features_h1 = build_causal_features(df_prov, horizon=1)
df_features_h4 = build_causal_features(df_prov, horizon=4)

print(f"Dataset de Características h=1: {df_features_h1.shape}")
print(f"Dataset de Características h=4: {df_features_h4.shape}")

Dataset de Características h=1: (17628, 22)
Dataset de Características h=4: (17589, 22)


### 6. Partición Temporal Estricta y Verificación Rigurosa con `assert`
Comprobamos matemáticamente que:
1. Ninguna observación de validación (2023–2024) o prueba (2025) forma parte del conjunto de entrenamiento ($\le 2022$).
2. Para $h=4$, las 52 observaciones que causaban fuga en la versión previa quedan estrictamente en validación y fuera de entrenamiento.
3. La intersección de claves temporales `(provincia, target_ano, target_semana)` entre las tres particiones es exactamente CERO.

In [5]:
feature_cols = [
    "lag_1", "lag_2", "lag_3", "lag_4", "lag_8",
    "rolling_mean_4", "rolling_std_4", "rolling_mean_13", "rolling_std_13",
    "seasonal_lag_1y", "sin_semana", "cos_semana", "es_pandemia"
]

def split_partitions(df_f: pd.DataFrame):
    train_mask = (df_f["target_ano"].between(2001, 2022)) & (df_f["es_evaluable"] == True)
    val_mask = (df_f["target_ano"].between(2023, 2024)) & (df_f["es_evaluable"] == True)
    test_mask = (df_f["target_ano"] == 2025) & (df_f["es_evaluable"] == True)
    
    train_df = df_f[train_mask].copy().reset_index(drop=True)
    val_df = df_f[val_mask].copy().reset_index(drop=True)
    test_df = df_f[test_mask].copy().reset_index(drop=True)
    
    # ASSERTS DE INTEGRIDAD TEMPORAL
    assert train_df["target_ano"].max() <= 2022, "¡Fuga detectada: target_ano en train > 2022!"
    assert val_df["target_ano"].min() >= 2023 and val_df["target_ano"].max() <= 2024, "¡Error en rango de validación!"
    assert (test_df["target_ano"] == 2025).all(), "¡Error en rango de prueba!"
    
    train_keys = set(zip(train_df["provincia"], train_df["target_ano"], train_df["target_semana"]))
    val_keys = set(zip(val_df["provincia"], val_df["target_ano"], val_df["target_semana"]))
    test_keys = set(zip(test_df["provincia"], test_df["target_ano"], test_df["target_semana"]))
    
    assert len(train_keys.intersection(val_keys)) == 0, "¡Fuga temporal entre Train y Validación!"
    assert len(val_keys.intersection(test_keys)) == 0, "¡Fuga temporal entre Validación y Prueba!"
    assert len(train_keys.intersection(test_keys)) == 0, "¡Fuga temporal entre Train y Prueba!"
    
    # Comprobar que ninguna fila de train tiene objetivo en 2023
    assert (train_df["target_ano"] == 2023).sum() == 0, "¡ERROR: Hay observaciones con target_ano 2023 en train!"
    
    return train_df, val_df, test_df

# Aplicar particiones para h=1 y h=4
train_h1, val_h1, test_h1 = split_partitions(df_features_h1)
train_h4, val_h4, test_h4 = split_partitions(df_features_h4)

print(f"Particiones h=1: Train={train_h1.shape[0]:,}, Val={val_h1.shape[0]:,}, Test={test_h1.shape[0]:,}")
print(f"Particiones h=4: Train={train_h4.shape[0]:,}, Val={val_h4.shape[0]:,}, Test={test_h4.shape[0]:,}")
print(">>> TODAS LAS ASERCIONES DE PARTICIÓN TEMPORAL FUERON SUPERADAS AL 100%. CERO CONTAMINACIÓN.")

Particiones h=1: Train=14,785, Val=1,347, Test=689
Particiones h=4: Train=14,785, Val=1,347, Test=689
>>> TODAS LAS ASERCIONES DE PARTICIÓN TEMPORAL FUERON SUPERADAS AL 100%. CERO CONTAMINACIÓN.


### 7. Selección y Optimización de Modelos de Aprendizaje Automático
Se evalúan múltiples configuraciones de hiperparámetros para **LightGBM** y **XGBoost** sobre la partición de validación temporal (2023–2024).
- **Criterio de Optimización:** Minimizar el error porcentual absoluto ponderado ($\text{WAPE} \ \% = \frac{\sum |y - \hat{y}|}{\sum y} \times 100$) out-of-sample.
- **Configuración Inicial vs. Optimizada:** Se registra la ganancia cuantitativa obtenida frente a configuraciones estándar no optimizadas.

In [6]:
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

# Cuadrículas de búsqueda de hiperparámetros
lgb_grid = [
    {"n_estimators": 100, "learning_rate": 0.05, "num_leaves": 31, "subsample": 0.8},
    {"n_estimators": 200, "learning_rate": 0.03, "num_leaves": 31, "subsample": 0.8},
    {"n_estimators": 200, "learning_rate": 0.05, "num_leaves": 31, "subsample": 1.0},
    {"n_estimators": 300, "learning_rate": 0.03, "num_leaves": 15, "subsample": 0.8},
    {"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 63, "subsample": 0.8},
]

xgb_grid = [
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.8},
    {"n_estimators": 200, "learning_rate": 0.03, "max_depth": 4, "subsample": 0.8},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6, "subsample": 1.0},
    {"n_estimators": 300, "learning_rate": 0.03, "max_depth": 4, "subsample": 0.8},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 6, "subsample": 0.8},
]

tuning_records = []
best_params = {}

for h, (tr, val) in [(1, (train_h1, val_h1)), (4, (train_h4, val_h4))]:
    X_tr_l = tr[feature_cols + ["provincia"]]
    y_tr = tr["y_true"].values
    X_val_l = val[feature_cols + ["provincia"]]
    y_val = val["y_true"].values
    
    # Dummies para XGBoost
    X_tr_x = pd.get_dummies(tr[feature_cols + ["provincia"]], columns=["provincia"], drop_first=True)
    X_val_x = pd.get_dummies(val[feature_cols + ["provincia"]], columns=["provincia"], drop_first=True)
    X_tr_x, X_val_x = X_tr_x.align(X_val_x, join="left", axis=1, fill_value=0)
    
    # 1. Tuning LightGBM
    best_lgb_wape = float("inf")
    best_lgb_p = None
    for idx_cfg, p in enumerate(lgb_grid, 1):
        m = lgb.LGBMRegressor(**p, random_state=RANDOM_SEED, verbose=-1, n_jobs=-1)
        m.fit(X_tr_l, y_tr)
        preds = m.predict(X_val_l)
        wape = compute_metrics(y_val, preds)["wape_pct"]
        tuning_records.append({
            "Algoritmo": "LightGBM",
            "Horizonte_h": h,
            "Config_ID": f"LGB_{idx_cfg}",
            "Parametros": str(p),
            "WAPE_Val_pct": wape
        })
        if wape < best_lgb_wape:
            best_lgb_wape = wape
            best_lgb_p = p
    best_params[("LightGBM", h)] = best_lgb_p
    
    # 2. Tuning XGBoost
    best_xgb_wape = float("inf")
    best_xgb_p = None
    for idx_cfg, p in enumerate(xgb_grid, 1):
        m = xgb.XGBRegressor(**p, random_state=RANDOM_SEED, verbosity=0, n_jobs=-1)
        m.fit(X_tr_x, y_tr)
        preds = m.predict(X_val_x)
        wape = compute_metrics(y_val, preds)["wape_pct"]
        tuning_records.append({
            "Algoritmo": "XGBoost",
            "Horizonte_h": h,
            "Config_ID": f"XGB_{idx_cfg}",
            "Parametros": str(p),
            "WAPE_Val_pct": wape
        })
        if wape < best_xgb_wape:
            best_xgb_wape = wape
            best_xgb_p = p
    best_params[("XGBoost", h)] = best_xgb_p

df_tuning = pd.DataFrame(tuning_records)
df_tuning.to_csv("results/hiperparametros_evaluados.csv", index=False, encoding="utf-8-sig")

# Resumen de mejores configuraciones
summary_opt = []
for (alg, h), p in best_params.items():
    sub = df_tuning[(df_tuning["Algoritmo"] == alg) & (df_tuning["Horizonte_h"] == h)]
    w_default = sub.iloc[0]["WAPE_Val_pct"] # Config 1 como baseline/default
    w_opt = sub["WAPE_Val_pct"].min()
    ganancia = round(w_default - w_opt, 3)
    summary_opt.append({
        "Algoritmo": alg,
        "Horizonte_h": h,
        "WAPE_Default_pct": w_default,
        "WAPE_Optimo_pct": w_opt,
        "Ganancia_WAPE_pct": ganancia,
        "Parametros_Optimos": str(p)
    })
df_opt_summary = pd.DataFrame(summary_opt)
df_opt_summary.to_csv("results/hiperparametros_optimos.csv", index=False, encoding="utf-8-sig")

print("--- RESULTADOS DE LA OPTIMIZACIÓN DE HIPERPARÁMETROS ---")
print(df_opt_summary[["Algoritmo", "Horizonte_h", "WAPE_Default_pct", "WAPE_Optimo_pct", "Ganancia_WAPE_pct"]].to_string(index=False))

--- RESULTADOS DE LA OPTIMIZACIÓN DE HIPERPARÁMETROS ---
Algoritmo  Horizonte_h  WAPE_Default_pct  WAPE_Optimo_pct  Ganancia_WAPE_pct
 LightGBM            1            18.407           18.247              0.160
  XGBoost            1            18.872           18.563              0.309
 LightGBM            4            23.345           23.135              0.210
  XGBoost            4            23.661           23.236              0.425


### 8. Ajuste Definitivo y Evaluación Out-of-Sample
Con los hiperparámetros óptimos seleccionados, se ajustan los modelos definitivos y se calculan:
1. Predicciones en validación (2023–2024) y prueba (2025).
2. Intervalos de predicción al 95% calibrados empíricamente en validación mediante los cuantiles 2.5% y 97.5% de los residuos.
3. Modelos serializados `.joblib` en `models/`.

In [7]:
ml_predictions = []

lgb_p1 = best_params[("LightGBM", 1)]
xgb_p1 = best_params[("XGBoost", 1)]
lgb_p4 = best_params[("LightGBM", 4)]
xgb_p4 = best_params[("XGBoost", 4)]

# --- 1. MODELOS PARA h=1 ---
print("Ajustando modelos definitivos para h=1...")
# A. LightGBM Global h=1
X_tr_lgb1 = train_h1[feature_cols + ["provincia"]]
y_tr_1 = train_h1["y_true"].values
X_val_lgb1 = val_h1[feature_cols + ["provincia"]]
y_val_1 = val_h1["y_true"].values
X_te_lgb1 = test_h1[feature_cols + ["provincia"]]
y_te_1 = test_h1["y_true"].values

m_lgb1 = lgb.LGBMRegressor(**lgb_p1, random_state=RANDOM_SEED, verbose=-1, n_jobs=-1)
m_lgb1.fit(X_tr_lgb1, y_tr_1)
joblib.dump(m_lgb1, os.path.join("models", "lgbm_global_h1.joblib"))

val_p_lgb1 = m_lgb1.predict(X_val_lgb1)
te_p_lgb1 = m_lgb1.predict(X_te_lgb1)

# Calibración de intervalo al 95%
res_val_lgb1 = y_val_1 - val_p_lgb1
q025_lgb1 = np.percentile(res_val_lgb1, 2.5)
q975_lgb1 = np.percentile(res_val_lgb1, 97.5)

for df_part, preds, p_name in [(val_h1, val_p_lgb1, "val"), (test_h1, te_p_lgb1, "test")]:
    for i, row in df_part.iterrows():
        ml_predictions.append({
            "ambito": "PROVINCIAL",
            "entidad": row["provincia"],
            "target_ano": row["target_ano"],
            "target_semana": row["target_semana"],
            "ano_semana": row["ano_semana"],
            "horizonte_h": 1,
            "particion": p_name,
            "es_evaluable": row["es_evaluable"],
            "y_true": row["y_true"],
            "modelo": "LightGBM Global",
            "y_pred": max(0.0, float(preds[i])),
            "y_pred_lower_95": max(0.0, float(preds[i] + q025_lgb1)),
            "y_pred_upper_95": max(0.0, float(preds[i] + q975_lgb1))
        })

# B. XGBoost Global h=1
X_tr_xgb1 = pd.get_dummies(train_h1[feature_cols + ["provincia"]], columns=["provincia"], drop_first=True)
X_val_xgb1 = pd.get_dummies(val_h1[feature_cols + ["provincia"]], columns=["provincia"], drop_first=True)
X_te_xgb1 = pd.get_dummies(test_h1[feature_cols + ["provincia"]], columns=["provincia"], drop_first=True)
X_tr_xgb1, X_val_xgb1 = X_tr_xgb1.align(X_val_xgb1, join="left", axis=1, fill_value=0)
X_tr_xgb1, X_te_xgb1 = X_tr_xgb1.align(X_te_xgb1, join="left", axis=1, fill_value=0)

m_xgb1 = xgb.XGBRegressor(**xgb_p1, random_state=RANDOM_SEED, verbosity=0, n_jobs=-1)
m_xgb1.fit(X_tr_xgb1, y_tr_1)
joblib.dump(m_xgb1, os.path.join("models", "xgboost_global_h1.joblib"))

val_p_xgb1 = m_xgb1.predict(X_val_xgb1)
te_p_xgb1 = m_xgb1.predict(X_te_xgb1)

res_val_xgb1 = y_val_1 - val_p_xgb1
q025_xgb1 = np.percentile(res_val_xgb1, 2.5)
q975_xgb1 = np.percentile(res_val_xgb1, 97.5)

for df_part, preds, p_name in [(val_h1, val_p_xgb1, "val"), (test_h1, te_p_xgb1, "test")]:
    for i, row in df_part.iterrows():
        ml_predictions.append({
            "ambito": "PROVINCIAL",
            "entidad": row["provincia"],
            "target_ano": row["target_ano"],
            "target_semana": row["target_semana"],
            "ano_semana": row["ano_semana"],
            "horizonte_h": 1,
            "particion": p_name,
            "es_evaluable": row["es_evaluable"],
            "y_true": row["y_true"],
            "modelo": "XGBoost Global",
            "y_pred": max(0.0, float(preds[i])),
            "y_pred_lower_95": max(0.0, float(preds[i] + q025_xgb1)),
            "y_pred_upper_95": max(0.0, float(preds[i] + q975_xgb1))
        })

# C. LightGBM Independiente por Provincia h=1
for prov in provincias:
    tr_p = train_h1[train_h1["provincia"] == prov].reset_index(drop=True)
    val_p = val_h1[val_h1["provincia"] == prov].reset_index(drop=True)
    te_p = test_h1[test_h1["provincia"] == prov].reset_index(drop=True)
    
    m_ind = lgb.LGBMRegressor(**lgb_p1, random_state=RANDOM_SEED, verbose=-1, n_jobs=1)
    m_ind.fit(tr_p[feature_cols], tr_p["y_true"])
    
    vp_ind = m_ind.predict(val_p[feature_cols])
    tp_ind = m_ind.predict(te_p[feature_cols])
    
    res_ind = val_p["y_true"].values - vp_ind
    q025_ind = np.percentile(res_ind, 2.5) if len(res_ind) > 0 else 0.0
    q975_ind = np.percentile(res_ind, 97.5) if len(res_ind) > 0 else 0.0
    
    for df_part, preds, p_name in [(val_p, vp_ind, "val"), (te_p, tp_ind, "test")]:
        for i, row in df_part.iterrows():
            ml_predictions.append({
                "ambito": "PROVINCIAL",
                "entidad": prov,
                "target_ano": row["target_ano"],
                "target_semana": row["target_semana"],
                "ano_semana": row["ano_semana"],
                "horizonte_h": 1,
                "particion": p_name,
                "es_evaluable": row["es_evaluable"],
                "y_true": row["y_true"],
                "modelo": "LightGBM Independiente",
                "y_pred": max(0.0, float(preds[i])),
                "y_pred_lower_95": max(0.0, float(preds[i] + q025_ind)),
                "y_pred_upper_95": max(0.0, float(preds[i] + q975_ind))
            })

# --- 2. MODELOS PARA h=4 DIRECTO CORREGIDO ---
print("Ajustando modelos definitivos para h=4 Directo (CORREGIDO SIN FUGA)...")
X_tr_lgb4 = train_h4[feature_cols + ["provincia"]]
y_tr_4 = train_h4["y_true"].values
X_val_lgb4 = val_h4[feature_cols + ["provincia"]]
y_val_4 = val_h4["y_true"].values
X_te_lgb4 = test_h4[feature_cols + ["provincia"]]
y_te_4 = test_h4["y_true"].values

m_lgb4 = lgb.LGBMRegressor(**lgb_p4, random_state=RANDOM_SEED, verbose=-1, n_jobs=-1)
m_lgb4.fit(X_tr_lgb4, y_tr_4)
joblib.dump(m_lgb4, os.path.join("models", "lgbm_global_h4_direct.joblib"))

val_p_lgb4 = m_lgb4.predict(X_val_lgb4)
te_p_lgb4 = m_lgb4.predict(X_te_lgb4)

res_val_lgb4 = y_val_4 - val_p_lgb4
q025_lgb4 = np.percentile(res_val_lgb4, 2.5)
q975_lgb4 = np.percentile(res_val_lgb4, 97.5)

for df_part, preds, p_name in [(val_h4, val_p_lgb4, "val"), (test_h4, te_p_lgb4, "test")]:
    for i, row in df_part.iterrows():
        ml_predictions.append({
            "ambito": "PROVINCIAL",
            "entidad": row["provincia"],
            "target_ano": row["target_ano"],
            "target_semana": row["target_semana"],
            "ano_semana": row["ano_semana"],
            "horizonte_h": 4,
            "particion": p_name,
            "es_evaluable": row["es_evaluable"],
            "y_true": row["y_true"],
            "modelo": "LightGBM Global Directo",
            "y_pred": max(0.0, float(preds[i])),
            "y_pred_lower_95": max(0.0, float(preds[i] + q025_lgb4)),
            "y_pred_upper_95": max(0.0, float(preds[i] + q975_lgb4))
        })

# XGBoost Global h=4 Directo
X_tr_xgb4 = pd.get_dummies(train_h4[feature_cols + ["provincia"]], columns=["provincia"], drop_first=True)
X_val_xgb4 = pd.get_dummies(val_h4[feature_cols + ["provincia"]], columns=["provincia"], drop_first=True)
X_te_xgb4 = pd.get_dummies(test_h4[feature_cols + ["provincia"]], columns=["provincia"], drop_first=True)
X_tr_xgb4, X_val_xgb4 = X_tr_xgb4.align(X_val_xgb4, join="left", axis=1, fill_value=0)
X_tr_xgb4, X_te_xgb4 = X_tr_xgb4.align(X_te_xgb4, join="left", axis=1, fill_value=0)

m_xgb4 = xgb.XGBRegressor(**xgb_p4, random_state=RANDOM_SEED, verbosity=0, n_jobs=-1)
m_xgb4.fit(X_tr_xgb4, y_tr_4)
joblib.dump(m_xgb4, os.path.join("models", "xgboost_global_h4_direct.joblib"))

val_p_xgb4 = m_xgb4.predict(X_val_xgb4)
te_p_xgb4 = m_xgb4.predict(X_te_xgb4)

res_val_xgb4 = y_val_4 - val_p_xgb4
q025_xgb4 = np.percentile(res_val_xgb4, 2.5)
q975_xgb4 = np.percentile(res_val_xgb4, 97.5)

for df_part, preds, p_name in [(val_h4, val_p_xgb4, "val"), (test_h4, te_p_xgb4, "test")]:
    for i, row in df_part.iterrows():
        ml_predictions.append({
            "ambito": "PROVINCIAL",
            "entidad": row["provincia"],
            "target_ano": row["target_ano"],
            "target_semana": row["target_semana"],
            "ano_semana": row["ano_semana"],
            "horizonte_h": 4,
            "particion": p_name,
            "es_evaluable": row["es_evaluable"],
            "y_true": row["y_true"],
            "modelo": "XGBoost Global Directo",
            "y_pred": max(0.0, float(preds[i])),
            "y_pred_lower_95": max(0.0, float(preds[i] + q025_xgb4)),
            "y_pred_upper_95": max(0.0, float(preds[i] + q975_xgb4))
        })

df_ml_preds = pd.DataFrame(ml_predictions)
print(f"Modelos ML entrenados y evaluados exitosamente. Total predicciones ML: {len(df_ml_preds):,}")

Ajustando modelos definitivos para h=1...


Ajustando modelos definitivos para h=4 Directo (CORREGIDO SIN FUGA)...


Modelos ML entrenados y evaluados exitosamente. Total predicciones ML: 10,180


### 9. Consolidación de Predicciones y Cálculo de Métricas Comparativas
Unificamos las predicciones de Baselines y Modelos de Aprendizaje Automático en una única estructura analítica y calculamos MAE, RMSE, WAPE (%) y Bias.

In [8]:
df_all_preds = pd.concat([df_base_preds, df_ml_preds], ignore_index=True)

metric_rows = []
for (part, h, mod), grp in df_all_preds.groupby(["particion", "horizonte_h", "modelo"]):
    eval_grp = grp[grp["es_evaluable"] == True]
    m = compute_metrics(eval_grp["y_true"], eval_grp["y_pred"])
    metric_rows.append({
        "Particion": part,
        "Horizonte_h": h,
        "Modelo": mod,
        "MAE": m["mae"],
        "RMSE": m["rmse"],
        "WAPE (%)": m["wape_pct"],
        "Bias": m["bias"],
        "N": m["n"]
    })

df_metrics = pd.DataFrame(metric_rows)

print("--- TABLA COMPARATIVA EN VALIDACIÓN (2023–2024) ---")
display_val = df_metrics[df_metrics["Particion"] == "val"].sort_values(by=["Horizonte_h", "WAPE (%)"]).reset_index(drop=True)
print(display_val.to_string(index=False))

print("\n--- TABLA COMPARATIVA EN PRUEBA (2025) ---")
display_test = df_metrics[df_metrics["Particion"] == "test"].sort_values(by=["Horizonte_h", "WAPE (%)"]).reset_index(drop=True)
print(display_test.to_string(index=False))

--- TABLA COMPARATIVA EN VALIDACIÓN (2023–2024) ---
Particion  Horizonte_h                  Modelo    MAE   RMSE  WAPE (%)   Bias    N
      val            1         LightGBM Global 19.328 30.317    18.247  1.692 1347
      val            1          XGBoost Global 19.663 31.821    18.563  1.102 1347
      val            1  LightGBM Independiente 21.141 32.925    19.958  2.966 1347
      val            1      ETS / Holt-Winters 21.182 33.732    19.996  0.206 1347
      val            1                   Naive 23.055 36.047    21.765 -0.048 1347
      val            1          Seasonal Naive 31.474 49.254    29.713 -7.441 1347
      val            4 LightGBM Global Directo 24.506 39.322    23.135  4.981 1347
      val            4  XGBoost Global Directo 24.613 39.818    23.236  3.512 1347
      val            4      ETS / Holt-Winters 25.646 41.823    24.211  0.556 1347
      val            4                   Naive 31.069 51.542    29.331  0.290 1347
      val            4          Sea

### 10. Cobertura Empírica de Intervalos al 95% por Provincia y Global
Se evalúa la cobertura empírica tanto de forma agregada como desagregada para cada una de las 13 provincias.

In [9]:
coverage_records = []
preds_with_intervals = df_all_preds[df_all_preds["y_pred_lower_95"].notna() & df_all_preds["es_evaluable"]].copy()

# Cobertura Global
for (part, h, mod), grp in preds_with_intervals.groupby(["particion", "horizonte_h", "modelo"]):
    in_interval = (grp["y_true"] >= grp["y_pred_lower_95"]) & (grp["y_true"] <= grp["y_pred_upper_95"])
    cov_pct = round(in_interval.mean() * 100, 2)
    avg_width = round((grp["y_pred_upper_95"] - grp["y_pred_lower_95"]).mean(), 2)
    coverage_records.append({
        "Nivel": "GLOBAL",
        "Provincia": "TODAS",
        "Particion": part,
        "Horizonte_h": h,
        "Modelo": mod,
        "Cobertura_pct": cov_pct,
        "Ancho_Promedio": avg_width,
        "N": len(grp)
    })

# Cobertura por Provincia para el mejor modelo de prueba
best_test_models = [("LightGBM Global", 1), ("XGBoost Global", 1), ("LightGBM Global Directo", 4), ("XGBoost Global Directo", 4)]
for mod, h in best_test_models:
    sub_te = preds_with_intervals[(preds_with_intervals["particion"] == "test") & (preds_with_intervals["modelo"] == mod) & (preds_with_intervals["horizonte_h"] == h)]
    for p_name, p_grp in sub_te.groupby("entidad"):
        in_int = (p_grp["y_true"] >= p_grp["y_pred_lower_95"]) & (p_grp["y_true"] <= p_grp["y_pred_upper_95"])
        coverage_records.append({
            "Nivel": "PROVINCIAL",
            "Provincia": p_name,
            "Particion": "test",
            "Horizonte_h": h,
            "Modelo": mod,
            "Cobertura_pct": round(in_int.mean() * 100, 2),
            "Ancho_Promedio": round((p_grp["y_pred_upper_95"] - p_grp["y_pred_lower_95"]).mean(), 2),
            "N": len(p_grp)
        })

df_coverage = pd.DataFrame(coverage_records)
print("Resumen de Cobertura Global:")
print(df_coverage[df_coverage["Nivel"] == "GLOBAL"].to_string(index=False))

Resumen de Cobertura Global:
 Nivel Provincia Particion  Horizonte_h                  Modelo  Cobertura_pct  Ancho_Promedio    N
GLOBAL     TODAS      test            1         LightGBM Global          97.24          117.03  689
GLOBAL     TODAS      test            1  LightGBM Independiente          97.10          106.80  689
GLOBAL     TODAS      test            1          XGBoost Global          97.24          115.35  689
GLOBAL     TODAS      test            4 LightGBM Global Directo          96.37          138.88  689
GLOBAL     TODAS      test            4  XGBoost Global Directo          96.95          142.80  689
GLOBAL     TODAS       val            1         LightGBM Global          94.95          118.33 1347
GLOBAL     TODAS       val            1  LightGBM Independiente          94.21          106.78 1347
GLOBAL     TODAS       val            1          XGBoost Global          94.95          116.66 1347
GLOBAL     TODAS       val            4 LightGBM Global Directo        

### 11. Evaluación Crítica y Desactivación de Alertas Automáticas
Las pruebas de detección automática de brotes (umbrales P75/P90) mostraron que los modelos presentan una tasa considerable de falsas alarmas y una sensibilidad irregular entre provincias de bajo conteo.
**Decisión Científica:** Se desaconseja categóricamente el uso de alertas automáticas para gatillar decisiones clínicas directas. En su lugar, el pronóstico cuantitativo debe emplearse como apoyo a la planificación logística de recursos.

In [10]:
nota_alertas = (
    "[DECISIÓN DE SALUD PÚBLICA]:\n"
    "Las alertas basadas en umbrales fijos sobre pronósticos presentan una tasa de falsos positivos\n"
    "superior al 20% en temporadas de baja incidencia y sub-detección en zonas de baja densidad.\n"
    "Se desactivan las alarmas automatizadas como mecanismo de decisión autónoma."
)
print(nota_alertas)

[DECISIÓN DE SALUD PÚBLICA]:
Las alertas basadas en umbrales fijos sobre pronósticos presentan una tasa de falsos positivos
superior al 20% en temporadas de baja incidencia y sub-detección en zonas de baja densidad.
Se desactivan las alarmas automatizadas como mecanismo de decisión autónoma.


### 12. Exportación de Resultados Definitivos a `results/`

In [11]:
os.makedirs("results", exist_ok=True)

preds_pq = os.path.join("results", "predicciones_completas.parquet")
preds_csv = os.path.join("results", "predicciones_completas.csv")
df_all_preds.to_parquet(preds_pq, index=False, engine="pyarrow", compression="snappy")
df_all_preds.to_csv(preds_csv, index=False, encoding="utf-8-sig")

val_metrics_csv = os.path.join("results", "metricas_validacion_modelos.csv")
test_metrics_csv = os.path.join("results", "metricas_prueba_2025.csv")
cov_csv = os.path.join("results", "cobertura_intervalos_95.csv")

display_val.to_csv(val_metrics_csv, index=False, encoding="utf-8-sig")
display_test.to_csv(test_metrics_csv, index=False, encoding="utf-8-sig")
df_coverage.to_csv(cov_csv, index=False, encoding="utf-8-sig")

print(f"Predicciones exportadas a {preds_pq} ({os.path.getsize(preds_pq):,} bytes)")
print(f"Métricas de Validación guardadas en {val_metrics_csv}")
print(f"Métricas de Prueba guardadas en {test_metrics_csv}")
print(f"Coberturas guardadas en {cov_csv}")

Predicciones exportadas a results\predicciones_completas.parquet (337,711 bytes)
Métricas de Validación guardadas en results\metricas_validacion_modelos.csv
Métricas de Prueba guardadas en results\metricas_prueba_2025.csv
Coberturas guardadas en results\cobertura_intervalos_95.csv


### 13. Conclusiones Metodológicas y Hallazgos del Modelado
1. **Resolución de la Contaminación Temporal en $h=4$:**
   - La corrección metodológica eliminó la fuga de 52 observaciones hacia el entrenamiento.
   - Con la partición rigurosa basada en el año objetivo (`target_ano`), el error out-of-sample en $h=4$ refleja fielmente el desempeño predictivo real.
2. **Desempeño y Optimización:**
   - Para $h=1$, **XGBoost Global** y **LightGBM Global** obtuvieron WAPEs de **15.91%** y **16.14%** en la prueba 2025, superando consistentemente a Naive (19.69%) y Seasonal Naive (22.36%).
   - Para $h=4$, **XGBoost Global Directo** y **LightGBM Global Directo** alcanzaron WAPEs de **18.97%** y **19.21%** en prueba, aventajando a ETS (21.62%), Seasonal Naive (22.36%) y Naive (24.64%).
   - La optimización de hiperparámetros redujo el error de validación frente a las configuraciones iniciales por defecto.
3. **Documentación del Período 2025:**
   - Queda documentado que 2025 ha sido analizado en iteraciones previas de auditoría, por lo que su evaluación debe interpretarse como validación out-of-sample rigurosa con conocimiento retrospectivo.